# Console de Ferramentas de Limpeza Sistêmica AWS

Comandos de expurgo forçado de emergência na rede em que o AgentCore executou provisionamento.
Execute o snippet Python ou Boto correspondente no alvo isolado AWS, ou use o modo **Tear Down / Nuke All** para deleção maciça em árvore.

> **ATENÇÃO TÉCNICA E CUSTOS (Warning):** Estas rotinas REST chamam exclusão absoluta e são irreversíveis. Em caso de infraestrutura legada e perdas você precisará iterar os módulos 2 ao 7 novamente rodando os arquivos Python da pasta do laboratório de engenharia na íntegra.

In [ ]:
import sys; sys.path.insert(0, '..')
import boto3
import time
from botocore.exceptions import ClientError
from shared import utils

region = utils.get_region()
control = boto3.client('bedrock-agentcore-control', region_name=region)
print(f'Region: {region}')
print(f'Account: {utils.get_account_id()}')
print('Ready.')

---
## Etapa 1. Apagar e Deletar Contêiner Runtime AWS

Ação sobre a base `aria_agent`. O Agente inteiro, seu pipeline, as microVM isoladas por id ativas e tudo do Strands será desconectado e eliminado do cluster na nuvem.

In [ ]:
print('Searching for aria runtimes...')
deleted = 0
paginator = control.get_paginator('list_agent_runtimes')
for page in paginator.paginate():
    for rt in page.get('agentRuntimes', page.get('agentRuntimeSummaries', [])):
        name = rt.get('agentRuntimeName', '')
        if 'aria' in name.lower():
            rt_id = rt['agentRuntimeId']
            print(f'  Deleting runtime: {name} ({rt_id})')
            try:
                control.delete_agent_runtime(agentRuntimeId=rt_id)
                # Wait for deletion
                for i in range(30):
                    try:
                        r = control.get_agent_runtime(agentRuntimeId=rt_id)
                        print(f'    Waiting... ({r.get("status", "?")})')
                        time.sleep(10)
                    except ClientError as e:
                        if 'ResourceNotFoundException' in str(e):
                            break
                        raise
                print(f'  Deleted: {name}')
                deleted += 1
            except ClientError as e:
                if 'ResourceNotFoundException' in str(e):
                    print(f'  Already deleted: {name}')
                else:
                    print(f'  Error: {e}')

# Clean up saved config
if deleted > 0:
    config_file = utils.CONFIG_DIR / 'runtime.json'
    if config_file.exists():
        config_file.unlink()
        print('  Removed saved runtime config')

print(f'\nDone. Deleted {deleted} runtime(s).')

---
## Etapa 2. Aniquilar Memory Database AWS

Comando de desmanche AWS no componente de Infra `AriaMemory` com purga completa LTM / STM corporativa atrelada (DynamoDB wrapper).

In [ ]:
print('Searching for aria memories...')
deleted = 0
paginator = control.get_paginator('list_memories')
for page in paginator.paginate():
    for mem in page.get('memories', page.get('items', [])):
        mem_id = mem['id']
        if mem_id.startswith('AriaMemory') or 'aria' in mem_id.lower():
            print(f'  Deleting memory: {mem_id}')
            try:
                control.delete_memory(memoryId=mem_id)
                for i in range(30):
                    try:
                        control.get_memory(memoryId=mem_id)
                        print(f'    Waiting...')
                        time.sleep(10)
                    except ClientError as e:
                        if 'ResourceNotFoundException' in str(e):
                            break
                        raise
                print(f'  Deleted: {mem_id}')
                deleted += 1
            except ClientError as e:
                if 'ResourceNotFoundException' in str(e):
                    print(f'  Already deleted: {mem_id}')
                else:
                    print(f'  Error: {e}')

if deleted > 0:
    config_file = utils.CONFIG_DIR / 'memory.json'
    if config_file.exists():
        config_file.unlink()
        print('  Removed saved memory config')

print(f'\nDone. Deleted {deleted} memory resource(s).')

---
## Etapa 3. Encerrar Roteador do AWS Gateway MCP

Anular instâncias associadas às tabelas e roteamento restrito da VPC interna que ligavam a API ao Agent. Os targets desaparecerão.

In [ ]:
print('Searching for aria gateways...')
deleted = 0
paginator = control.get_paginator('list_gateways')
for page in paginator.paginate():
    for gw in page.get('items', []):
        name = gw.get('name', '')
        if 'aria' in name.lower():
            gw_id = gw['gatewayId']
            print(f'  Found gateway: {name} ({gw_id})')

            # Delete targets first
            try:
                targets = control.list_gateway_targets(gatewayIdentifier=gw_id)
                for t in targets.get('items', []):
                    t_id = t['targetId']
                    t_name = t.get('name', t_id)
                    print(f'    Deleting target: {t_name}')
                    control.delete_gateway_target(gatewayIdentifier=gw_id, targetId=t_id)
                    for i in range(30):
                        try:
                            control.get_gateway_target(gatewayIdentifier=gw_id, targetId=t_id)
                            time.sleep(10)
                        except ClientError:
                            break
                    print(f'    Deleted target: {t_name}')
            except ClientError as e:
                print(f'    Could not list/delete targets: {e}')

            # Delete gateway
            print(f'  Deleting gateway: {name}')
            try:
                control.delete_gateway(gatewayIdentifier=gw_id)
                for i in range(30):
                    try:
                        control.get_gateway(gatewayIdentifier=gw_id)
                        print(f'    Waiting...')
                        time.sleep(10)
                    except ClientError as e:
                        if 'ResourceNotFoundException' in str(e):
                            break
                        raise
                print(f'  Deleted: {name}')
                deleted += 1
            except ClientError as e:
                if 'ResourceNotFoundException' in str(e):
                    print(f'  Already deleted: {name}')
                else:
                    print(f'  Error: {e}')

if deleted > 0:
    config_file = utils.CONFIG_DIR / 'gateway.json'
    if config_file.exists():
        config_file.unlink()
        print('  Removed saved gateway config')

print(f'\nDone. Deleted {deleted} gateway(s).')

---
## Etapa 4. Interromper Mecanismo Restrito Policy Engine e Regras Cedar

Acionar cancelamento absoluto do módulo `aria-policy-engine`, com limpeza simultânea remota das cláusulas rígidas Cedar em IAM.

In [ ]:
print('Searching for aria policy engines...')
deleted = 0
paginator = control.get_paginator('list_policy_engines')
for page in paginator.paginate():
    for engine in page.get('policyEngines', page.get('items', [])):
        name = engine.get('name', '')
        if 'aria' in name.lower():
            eng_id = engine['policyEngineId']
            print(f'  Found policy engine: {name} ({eng_id})')

            # Delete policies first
            try:
                policies = control.list_policies(policyEngineId=eng_id)
                for p in policies.get('policies', policies.get('items', [])):
                    p_id = p['policyId']
                    p_name = p.get('name', p_id)
                    print(f'    Deleting policy: {p_name}')
                    control.delete_policy(policyEngineId=eng_id, policyId=p_id)
                    print(f'    Deleted: {p_name}')
            except ClientError as e:
                print(f'    Could not list/delete policies: {e}')

            # Delete engine
            print(f'  Deleting policy engine: {name}')
            try:
                control.delete_policy_engine(policyEngineId=eng_id)
                for i in range(30):
                    try:
                        control.get_policy_engine(policyEngineId=eng_id)
                        print(f'    Waiting...')
                        time.sleep(10)
                    except ClientError as e:
                        if 'ResourceNotFoundException' in str(e):
                            break
                        raise
                print(f'  Deleted: {name}')
                deleted += 1
            except ClientError as e:
                if 'ResourceNotFoundException' in str(e):
                    print(f'  Already deleted: {name}')
                else:
                    print(f'  Error: {e}')

if deleted > 0:
    config_file = utils.CONFIG_DIR / 'policy.json'
    if config_file.exists():
        config_file.unlink()
        print('  Removed saved policy config')

print(f'\nDone. Deleted {deleted} policy engine(s).')

---
## Etapa de Nuvem 5. Tear Down Integral (Apagar a Infra completa AWS)

Sequência autônoma orquestrada que elimina hierarquicamente respeitando proteções estritas (Policy > Gateway > Memory > e, finalmente o Contêiner Runtime global Python) mitigando erro interno na AWS Cloud.

In [ ]:
import subprocess, os

print('=' * 50)
print(' DELETING ALL AGENTCORE WORKSHOP RESOURCES')
print('=' * 50)
print()

# We re-run the cells above in order by executing them inline.
# Policy engine and gateway must go before runtime (gateway may reference runtime).

# --- Policy Engines ---
print('\n--- Policy Engines ---')
deleted_policy = 0
try:
    paginator = control.get_paginator('list_policy_engines')
    for page in paginator.paginate():
        for engine in page.get('policyEngines', page.get('items', [])):
            if 'aria' in engine.get('name', '').lower():
                eng_id = engine['policyEngineId']
                try:
                    policies = control.list_policies(policyEngineId=eng_id)
                    for p in policies.get('policies', policies.get('items', [])):
                        control.delete_policy(policyEngineId=eng_id, policyId=p['policyId'])
                        print(f'  Deleted policy: {p.get("name", p["policyId"])}')
                except ClientError:
                    pass
                control.delete_policy_engine(policyEngineId=eng_id)
                print(f'  Deleting policy engine: {engine["name"]} ({eng_id})')
                deleted_policy += 1
except ClientError as e:
    print(f'  {e}')
print(f'  {deleted_policy} policy engine(s) queued for deletion')

# --- Gateways ---
print('\n--- Gateways ---')
deleted_gw = 0
try:
    paginator = control.get_paginator('list_gateways')
    for page in paginator.paginate():
        for gw in page.get('items', []):
            if 'aria' in gw.get('name', '').lower():
                gw_id = gw['gatewayId']
                try:
                    targets = control.list_gateway_targets(gatewayIdentifier=gw_id)
                    for t in targets.get('items', []):
                        control.delete_gateway_target(gatewayIdentifier=gw_id, targetId=t['targetId'])
                        print(f'  Deleted target: {t.get("name", t["targetId"])}')
                except ClientError:
                    pass
                control.delete_gateway(gatewayIdentifier=gw_id)
                print(f'  Deleting gateway: {gw["name"]} ({gw_id})')
                deleted_gw += 1
except ClientError as e:
    print(f'  {e}')
print(f'  {deleted_gw} gateway(s) queued for deletion')

# --- Memories ---
print('\n--- Memories ---')
deleted_mem = 0
try:
    paginator = control.get_paginator('list_memories')
    for page in paginator.paginate():
        for mem in page.get('memories', page.get('items', [])):
            if mem['id'].startswith('AriaMemory') or 'aria' in mem['id'].lower():
                control.delete_memory(memoryId=mem['id'])
                print(f'  Deleting memory: {mem["id"]}')
                deleted_mem += 1
except ClientError as e:
    print(f'  {e}')
print(f'  {deleted_mem} memory resource(s) queued for deletion')

# --- Runtimes ---
print('\n--- Runtimes ---')
deleted_rt = 0
try:
    paginator = control.get_paginator('list_agent_runtimes')
    for page in paginator.paginate():
        for rt in page.get('agentRuntimes', page.get('agentRuntimeSummaries', [])):
            if 'aria' in rt.get('agentRuntimeName', '').lower():
                control.delete_agent_runtime(agentRuntimeId=rt['agentRuntimeId'])
                print(f'  Deleting runtime: {rt["agentRuntimeName"]} ({rt["agentRuntimeId"]})')
                deleted_rt += 1
except ClientError as e:
    print(f'  {e}')
print(f'  {deleted_rt} runtime(s) queued for deletion')

# --- Clean up saved configs ---
print('\n--- Config files ---')
for name in ['runtime', 'memory', 'gateway', 'policy', 'frontend']:
    config_file = utils.CONFIG_DIR / f'{name}.json'
    if config_file.exists():
        config_file.unlink()
        print(f'  Removed {name}.json')

# --- Wait for everything to finish ---
print('\n--- Waiting for deletions to complete ---')
print('  (this may take a minute or two)')
time.sleep(30)

# Verify
remaining = []
try:
    for page in control.get_paginator('list_agent_runtimes').paginate():
        for rt in page.get('agentRuntimes', page.get('agentRuntimeSummaries', [])):
            if 'aria' in rt.get('agentRuntimeName', '').lower():
                remaining.append(f'runtime: {rt["agentRuntimeName"]} ({rt.get("status", "?")})')
except ClientError:
    pass

if remaining:
    print(f'\n  Still deleting ({len(remaining)} resources):')
    for r in remaining:
        print(f'    {r}')
    print('  These should finish shortly. Re-run this cell to check.')
else:
    print('\n  All resources deleted.')

print('\n' + '=' * 50)
print(' CLEANUP COMPLETE')
print('=' * 50)